# Integração multimodal
**Nirton Afonso de Oliveira Filho - RM 370221**

Integração dos resultados de áudio e vídeo do Tech Challenge Fase 4. O notebook gera um resultado por caso, relatório em Markdown e fila de revisão em JSONL.

Execute todas as células com os controles padrão para testar os exemplos simulados. Essa execução usa apenas a biblioteca padrão do Python e dispensa GPU.

| Modo | Configuração |
|---|---|
| Demonstração | Manter todos os controles desligados |
| CSV de áudio + JSON de vídeo | Ativar `EXECUTAR_CSV_REAL` e confirmar o pareamento |
| Teste de um WAV | Ativar `EXECUTAR_AUDIO_WAV` |
| Teste de um vídeo | Ativar `EXECUTAR_VIDEO` |
| Teste dos serviços Azure | Ativar `EXECUTAR_AZURE` e cadastrar os Secrets |
| Integração dos arquivos com fontes adicionais | Ativar `EXECUTAR_INTEGRACAO` e confirmar o pareamento |

Os testes de cada serviço são independentes da integração final. As saídas ficam em `/content/TechChallenge4/integracao/saidas`.

A fusão é feita por caso, sem sincronização temporal entre as fontes. Os níveis indicam prioridade de revisão; o protótipo processa arquivos em lote.


In [ ]:
from pathlib import Path
import csv, hashlib, io, json, math, os, re, statistics, sys, unicodedata, zipfile
from collections import Counter
from datetime import datetime, timezone
from copy import deepcopy

try:
    import google.colab
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
RAIZ = Path("/content/TechChallenge4/integracao") if EM_COLAB else Path.cwd() / "saida_notebook"
ENTRADAS = RAIZ / "entradas"
SAIDA = RAIZ / "saidas"
for pasta in (ENTRADAS, SAIDA):
    pasta.mkdir(parents=True, exist_ok=True)

# Deixe os controles desligados para executar a demonstração sem serviços externos.
BAIXAR_AUDIO_DRIVE = False
BAIXAR_DATASET_LFS = False
EXECUTAR_CSV_REAL = False
EXECUTAR_AUDIO_WAV = False
EXECUTAR_VIDEO = False
EXECUTAR_AZURE = False
EXECUTAR_INTEGRACAO = False

CSV_AUDIO = ENTRADAS / "resultado_audio.csv"
MODELO_AUDIO = ENTRADAS / "modelo_audio.joblib"
WAV_AUDIO = ENTRADAS / "consulta.wav"
JSON_VIDEO = ENTRADAS / "resultado_video.json"
VIDEO = ENTRADAS / "cirurgia.mp4"
PESOS_VIDEO = ENTRADAS / "best_uterus_tools_v4.pt"

# Se o CSV contiver um lote, escolha um nome EXATO após inspecionar a tabela.
ARQUIVO_AUDIO_NO_CSV = "ansiedade_hesitacao_000_original.wav"
ID_CASO_REAL = "caso-pareado-001"
PAREAMENTO_CONFIRMADO = False  # marque True após confirmar o caso dos dois arquivos
LIMIAR_AUDIO = 0.70           # heurística de demonstração, ainda não calibrada
COMMIT_BASE = "8b6a06cd1456ade14b18ef0aa8096e49801f62fb"
print("Pasta de saída:", SAIDA)


## 1. Contratos e fusão
O áudio pode chegar como tupla `(classe, confianca)` ou como linha do CSV, com as chaves `classe_audio` e `confianca_audio`. O resultado de vídeo usa `alerta`, `nivel` e `eventos`, conforme o módulo da frente de vídeo.

A regra de fusão considera qualquer sinal válido e preserva a maior prioridade. Fontes ausentes, com erro ou confiança abaixo do limiar ficam inconclusivas. As confianças dos modelos permanecem separadas.

O identificador do caso e o pareamento dos arquivos devem ser informados. Áudios sintéticos e vídeos independentes podem ser usados na demonstração, com a indicação de simulação.


In [ ]:
NIVEIS = {"nenhum": 0, "baixo": 1, "medio": 2, "alto": 3}
CLASSES_AUDIO = {"ansiedade_hesitacao", "neutro", "tristeza"}
STATUS_FONTE = {"ok", "baixa_confianca", "indisponivel", "erro"}
AVISO = "Protótipo acadêmico para revisão humana. Não constitui diagnóstico."

def numero_finito(valor, nome, minimo=None, maximo=None):
    if isinstance(valor, bool):
        raise ValueError(f"{nome}: booleano não é um número válido.")
    x = float(valor)
    if not math.isfinite(x) or (minimo is not None and x < minimo) or (
        maximo is not None and x > maximo
    ):
        raise ValueError(f"{nome}: valor fora do intervalo permitido.")
    return x

def fonte_indisponivel(modalidade, motivo, status="indisponivel", simulado=False):
    return {"modalidade": modalidade, "status": status, "alerta": None,
            "nivel": "nenhum", "motivo": str(motivo), "simulado": bool(simulado)}

def normalizar_audio(resultado, origem="classificar_audio", simulado=False, limiar=LIMIAR_AUDIO):
    if resultado is None:
        return fonte_indisponivel("audio", "Áudio não fornecido.", simulado=simulado)
    if isinstance(resultado, dict):
        simulado = bool(simulado or resultado.get("simulado", False))
    limiar = numero_finito(limiar, "limiar", 0, 1)
    if isinstance(resultado, (tuple, list)) and len(resultado) == 2:
        classe, confianca = resultado
    elif isinstance(resultado, dict):
        if resultado.get("status") in {"erro", "indisponivel"}:
            return fonte_indisponivel("audio", resultado.get("motivo", "Falha no áudio."),
                                     resultado["status"], simulado)
        classe = resultado.get("classe_audio", resultado.get("classe"))
        confianca = resultado.get("confianca_audio", resultado.get("confianca"))
    else:
        raise ValueError("Áudio: use tupla (classe, confiança) ou uma linha do CSV.")
    if classe not in CLASSES_AUDIO:
        raise ValueError(f"Classe de áudio desconhecida: {classe!r}")
    p = numero_finito(confianca, "confianca_audio", 0, 1)
    suficiente = p >= limiar
    sinal = suficiente and classe != "neutro"
    return {"modalidade": "audio", "status": "ok" if suficiente else "baixa_confianca",
            "alerta": bool(sinal), "nivel": "medio" if sinal else "nenhum",
            "classe_audio": classe, "confianca_audio": p, "limiar": limiar,
            "motivo": f"Classe acústica: {classe}; confiança {p:.3f}.",
            "origem": origem, "simulado": bool(simulado)}

def normalizar_video(resultado, origem="analisar_video", simulado=False):
    if resultado is None:
        return fonte_indisponivel("video", "Vídeo não fornecido.", simulado=simulado)
    if not isinstance(resultado, dict):
        raise ValueError("Vídeo: esperado um dicionário JSON.")
    simulado = bool(simulado or resultado.get("simulado", False))
    if resultado.get("status") in {"erro", "indisponivel"}:
        return fonte_indisponivel("video", resultado.get("motivo", "Falha no vídeo."),
                                 resultado["status"], simulado)
    alerta = resultado.get("alerta")
    if type(alerta) is not bool:
        raise ValueError("Vídeo: 'alerta' deve ser booleano JSON, nunca 'true' como texto.")
    nivel = resultado.get("nivel", "baixo" if alerta else "nenhum")
    if nivel not in NIVEIS or (alerta and nivel == "nenhum") or (
        not alerta and nivel != "nenhum"
    ):
        raise ValueError("Vídeo: alerta e nível são inconsistentes.")
    eventos = resultado.get("eventos", [])
    if not isinstance(eventos, list):
        raise ValueError("Vídeo: eventos deve ser uma lista.")
    for e in eventos:
        inicio = numero_finito(e["inicio_s"], "inicio_s", 0)
        fim = numero_finito(e["fim_s"], "fim_s", 0)
        if fim <= inicio:
            raise ValueError("Vídeo: evento com intervalo inválido.")
        if "duracao_s" in e and abs(numero_finito(e["duracao_s"], "duracao_s", 0) - (fim-inicio)) > 0.21:
            raise ValueError("Vídeo: duração inconsistente.")
        if "confianca" in e:
            numero_finito(e["confianca"], "confianca", 0, 1)
    if eventos and not alerta:
        raise ValueError("Vídeo: há eventos, mas alerta está desativado.")
    return {"modalidade": "video", "status": "ok", "alerta": alerta, "nivel": nivel,
            "motivo": str(resultado.get("motivo", "Resultado de vídeo recebido.")),
            "eventos": deepcopy(eventos), "estatisticas": deepcopy(resultado.get("estatisticas", {})),
            "origem": origem, "simulado": bool(simulado)}

def validar_fonte(fonte):
    if not isinstance(fonte, dict) or not isinstance(fonte.get("modalidade"), str):
        raise ValueError("Fonte adicional inválida.")
    if fonte.get("status") not in STATUS_FONTE or fonte.get("nivel") not in NIVEIS:
        raise ValueError("Fonte adicional com status ou nível inválido.")
    if fonte["status"] == "ok":
        if type(fonte.get("alerta")) is not bool:
            raise ValueError("Fonte disponível exige alerta booleano.")
        if fonte["alerta"] != (fonte["nivel"] != "nenhum"):
            raise ValueError("Fonte adicional com alerta e nível inconsistentes.")
    elif fonte.get("alerta") not in (None, False) or fonte["nivel"] != "nenhum":
        raise ValueError("Fonte inconclusiva não deve fornecer sinal negativo conclusivo ou positivo.")
    return deepcopy(fonte)

def combinar_audio_video(resultado_audio, resultado_video, *, id_caso,
                         origem_audio="classificar_audio", origem_video="analisar_video",
                         simulado=False, pareamento="associacao_manual_por_caso",
                         fontes_adicionais=None, fontes_esperadas=("audio", "video")):
    if not isinstance(id_caso, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,80}", id_caso):
        raise ValueError("Use id_caso com 1 a 80 letras, números, '_' ou '-'.")
    if not isinstance(pareamento, str) or not pareamento.strip():
        raise ValueError("Descreva o pareamento dos arquivos.")
    fontes = [
        normalizar_audio(resultado_audio, origem_audio, simulado),
        normalizar_video(resultado_video, origem_video, simulado),
    ] + [validar_fonte(f) for f in (fontes_adicionais or [])]
    nomes = [f["modalidade"] for f in fontes]
    if len(nomes) != len(set(nomes)):
        raise ValueError("Há modalidades duplicadas.")
    esperadas = list(dict.fromkeys(fontes_esperadas))
    if not esperadas or not {"audio", "video"}.issubset(esperadas):
        raise ValueError("O escopo mínimo esperado é áudio e vídeo.")
    for nome in esperadas:
        if nome not in nomes:
            fontes.append(fonte_indisponivel(nome, "Fonte prevista, mas não fornecida."))
    indisponiveis = [f["modalidade"] for f in fontes if f["status"] != "ok"]
    sinais = [f for f in fontes if f["status"] == "ok" and f["alerta"]]
    nivel = max((f["nivel"] for f in sinais), key=NIVEIS.get, default="nenhum")
    assinatura = json.dumps({"id_caso": id_caso, "fontes": fontes, "pareamento": pareamento},
                           ensure_ascii=False, sort_keys=True, allow_nan=False)
    return {
        "schema_version": "1.0",
        "id_caso": id_caso,
        "id_resultado": hashlib.sha256(assinatura.encode()).hexdigest()[:20],
        "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
        "alerta": bool(sinais), "nivel": nivel,
        "requer_revisao": bool(sinais or indisponiveis),
        "status": "incompleto" if indisponiveis else "completo",
        "simulado": bool(simulado or any(f.get("simulado", False) for f in fontes)),
        "pareamento": pareamento,
        "fontes_esperadas": esperadas,
        "fontes_inconclusivas": indisponiveis,
        "motivos": [f["motivo"] for f in sinais],
        "fontes": fontes,
        "entrega": {"destino": "fila_local_equipe_medica", "status": "registrado_localmente"},
        "aviso": AVISO,
    }

def relatorio_automatico(resultado):
    r = resultado
    linhas = ["# Relatório automático multimodal", "",
              f"**Caso:** {r['id_caso']}  ",
              f"**Identificador:** {r['id_resultado']}  ",
              f"**Origem simulada:** {'sim' if r['simulado'] else 'não'}  ",
              f"**Cobertura:** {r['status']}  ",
              f"**Alerta:** {r['alerta']} | **Prioridade:** {r['nivel']}  ",
              f"**Revisão necessária:** {r['requer_revisao']}  ",
              f"**Pareamento:** {r['pareamento']}", "",
              "## Evidências por fonte", ""]
    for f in r["fontes"]:
        linhas += [f"### {f['modalidade']}", "",
                   f"- Status: {f['status']}",
                   f"- Sinal: {f['alerta']} | Nível: {f['nivel']}",
                   f"- Motivo: {f['motivo']}"]
        for e in f.get("eventos", []):
            if "inicio_s" in e and "fim_s" in e:
                linhas.append(f"- Evento: {e['inicio_s']} a {e['fim_s']} s.")
        if f.get("termos_criticos"):
            linhas.append("- Termos para revisão: " + ", ".join(f["termos_criticos"]))
        linhas.append("")
    if r["fontes_inconclusivas"]:
        linhas += ["## Cobertura incompleta", "",
                   "Revisar as fontes: " + ", ".join(r["fontes_inconclusivas"]) + ".", ""]
    linhas += ["## Resposta operacional", "",
               "Resultado registrado em arquivo local para revisão humana. "
               "Este protótipo não confirma entrega a um sistema hospitalar.", "",
               "> " + AVISO, ""]
    return "\n".join(linhas)

def salvar_resultados(resultados, pasta):
    pasta = Path(pasta)
    pasta.mkdir(parents=True, exist_ok=True)
    ids = [r["id_caso"] for r in resultados]
    if len(ids) != len(set(ids)):
        raise ValueError("Cada lote deve ter id_caso único.")
    for r in resultados:
        (pasta / f"{r['id_caso']}.json").write_text(
            json.dumps(r, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
        (pasta / f"{r['id_caso']}.md").write_text(relatorio_automatico(r), encoding="utf-8")
    # Snapshot do lote: repetir a célula não duplica linhas.
    with (pasta / "resultados.jsonl").open("w", encoding="utf-8") as arq:
        for r in resultados:
            arq.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    with (pasta / "fila_revisao.jsonl").open("w", encoding="utf-8") as arq:
        for r in resultados:
            if r["requer_revisao"]:
                arq.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    with (pasta / "resumo.csv").open("w", newline="", encoding="utf-8") as arq:
        nomes = ["id_caso", "alerta", "nivel", "status", "simulado", "requer_revisao"]
        w = csv.DictWriter(arq, fieldnames=nomes)
        w.writeheader()
        w.writerows({k: r[k] for k in nomes} for r in resultados)
    return pasta


## 2. Demonstração
Quatro casos simulados: ausência de sinal, sinal acústico, sinal de vídeo com áudio neutro e áudio indisponível. O último caso deve pedir revisão por cobertura incompleta.


In [ ]:
VIDEO_SEM_SINAL = {"alerta": False, "nivel": "nenhum", "motivo": "Sem evento na simulação.", "eventos": []}
VIDEO_COM_SINAL = {
    "alerta": True, "nivel": "alto", "motivo": "Instrumento sem útero visível na simulação.",
    "eventos": [{"inicio_s": 12.0, "fim_s": 30.0, "duracao_s": 18.0, "confianca": 0.81}]
}
cenarios = [
    ("demo-01", ("neutro", 0.95), VIDEO_SEM_SINAL),
    ("demo-02", ("ansiedade_hesitacao", 0.81), VIDEO_SEM_SINAL),
    ("demo-03", ("neutro", 0.97), VIDEO_COM_SINAL),
    ("demo-04", None, VIDEO_SEM_SINAL),
]
resultados_demo = [
    combinar_audio_video(a, v, id_caso=i, simulado=True,
                         pareamento="demonstracao_sem_associacao_clinica_ou_temporal")
    for i, a, v in cenarios
]
salvar_resultados(resultados_demo, SAIDA / "demonstracao")
for r in resultados_demo:
    print(r["id_caso"], "| alerta:", r["alerta"], "| nível:", r["nivel"],
          "| cobertura:", r["status"], "| revisão:", r["requer_revisao"])
print("\n", relatorio_automatico(resultados_demo[2]))


## 3. Leitura do CSV e do JSON
O CSV de áudio deve conter `arquivo`, `classe_audio` e `confianca_audio`. A seleção exige o nome exato do arquivo e rejeita duplicatas.

Envie os arquivos pelo painel Arquivos do Colab e ajuste os caminhos na configuração. Para usar o Drive montado no Colab, informe os caminhos correspondentes em `/content/drive/MyDrive`.


In [ ]:
def ler_audio_csv(caminho, arquivo):
    with Path(caminho).open(encoding="utf-8-sig", newline="") as f:
        leitor = csv.DictReader(f)
        if not {"arquivo", "classe_audio", "confianca_audio"}.issubset(leitor.fieldnames or []):
            raise ValueError("CSV de áudio sem as três colunas esperadas.")
        linhas = list(leitor)
    nomes = [r["arquivo"] for r in linhas]
    if len(nomes) != len(set(nomes)):
        raise ValueError("CSV de áudio possui nomes duplicados.")
    achados = [r for r in linhas if r["arquivo"] == arquivo]
    if len(achados) != 1:
        raise ValueError(f"Arquivo não encontrado de forma única no CSV: {arquivo}")
    normalizar_audio(achados[0])  # valida classe, intervalo e valores não finitos
    return achados[0]

def integrar_csv_json(csv_audio, arquivo_audio, json_video, *, id_caso,
                      pareamento_confirmado=False):
    if not pareamento_confirmado:
        raise ValueError("Confirme o pareamento dos dois arquivos por caso antes de integrar.")
    audio = ler_audio_csv(csv_audio, arquivo_audio)
    video = json.loads(Path(json_video).read_text(encoding="utf-8-sig"))
    return combinar_audio_video(
        audio, video, id_caso=id_caso,
        origem_audio=f"CSV:{Path(csv_audio).name}:{arquivo_audio}",
        origem_video=f"JSON:{Path(json_video).name}",
        pareamento="arquivos_associados_explicitamente_por_caso_sem_sincronia_verificada",
    )



## 4. Download dos artefatos
`BAIXAR_AUDIO_DRIVE` obtém o modelo e o CSV compartilhados pela frente de áudio. Os hashes identificam a versão verificada; revise-os se os arquivos forem substituídos.

`BAIXAR_DATASET_LFS` obtém o ZIP do repositório, com aproximadamente 100 MB. A versão examinada contém 1.180 WAVs: 380 de ansiedade/hesitação, 400 neutros e 400 de tristeza. O CSV de resultados possui 1.200 linhas; a diferença está descrita no relatório técnico.


In [ ]:
HASH_MODELO_AUDIO = "c85f3b2aa3ff936bd10479102d6b55ce922ad77d893c283c6fd600efd57fbfe9"
HASH_CSV_AUDIO = "21a821253c7c36594f6248392acc26eb9796055e10e9a3ce8e98e17edeab1d69"
HASH_DATASET = "d37a28e118f648be967c37224d27f9a54cb1bdd76446be30033235d5f74f83e6"

def verificar_hash(caminho, esperado):
    with Path(caminho).open("rb") as f:
        obtido = hashlib.file_digest(f, "sha256").hexdigest()
    if obtido != esperado:
        raise ValueError(f"SHA-256 divergente em {Path(caminho).name}. Revise a versão do arquivo.")
    return obtido

def instalar(pacotes):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pacotes])

def baixar_drive(file_id, destino, sha256):
    import gdown
    destino = Path(destino)
    if not destino.exists():
        temporario = destino.with_suffix(destino.suffix + ".download")
        if not gdown.download(id=file_id, output=str(temporario), quiet=False):
            raise RuntimeError("Drive não disponibilizou o arquivo. Faça upload pelo painel Arquivos.")
        verificar_hash(temporario, sha256)
        temporario.replace(destino)
    verificar_hash(destino, sha256)

def auditar_zip(caminho):
    with zipfile.ZipFile(caminho) as z:
        nomes = z.namelist()
        metadados = [n for n in nomes if Path(n).name == "metadata.csv"]
        if len(metadados) != 1:
            raise ValueError("Dataset deve ter um único metadata.csv.")
        registros = list(csv.DictReader(io.StringIO(z.read(metadados[0]).decode("utf-8-sig"))))
        wavs = [n for n in nomes if n.lower().endswith(".wav")]
        nomes_wav = [Path(n).name for n in wavs]
        if len(nomes_wav) != len(set(nomes_wav)):
            raise ValueError("Há WAVs com nomes duplicados em subpastas.")
        faltantes = [r["arquivo"] for r in registros if r["arquivo"] not in nomes_wav]
        return {"wavs": len(wavs), "linhas_metadata": len(registros),
                "classes_metadata": dict(Counter(r["classe"] for r in registros)),
                "arquivos_metadata_ausentes": faltantes}

def extrair_wav(caminho_zip, nome, destino):
    # Extrai somente o WAV escolhido, sem confiar nos caminhos internos do ZIP.
    if Path(nome).name != nome or not nome.lower().endswith(".wav"):
        raise ValueError("Informe apenas o nome de um WAV.")
    with zipfile.ZipFile(caminho_zip) as z:
        candidatos = [n for n in z.namelist() if Path(n).name == nome]
        if len(candidatos) != 1:
            raise ValueError("WAV não localizado de forma única no ZIP.")
        Path(destino).write_bytes(z.read(candidatos[0]))
    return Path(destino)

if BAIXAR_AUDIO_DRIVE:
    instalar(["gdown>=5.2,<6"])
    baixar_drive("1medFLIM3qO4UvqqMTwHOq0jD-Ud6DtQA", MODELO_AUDIO, HASH_MODELO_AUDIO)
    baixar_drive("1kW7IBad_U4i7DT2xsVZAFSAXB3Ua0Bpm", CSV_AUDIO, HASH_CSV_AUDIO)
    print("Modelo e CSV disponíveis:", ENTRADAS)

if BAIXAR_DATASET_LFS:
    from urllib.request import urlretrieve
    caminho_zip = ENTRADAS / "dataset_audio_emocoes_1200.zip"
    if not caminho_zip.exists():
        url = f"https://media.githubusercontent.com/media/Val-Faria/TechChallengeFase4/{COMMIT_BASE}/audio/datasets/dataset_audio_emocoes_1200.zip"
        temporario = caminho_zip.with_suffix(".download")
        urlretrieve(url, temporario)
        verificar_hash(temporario, HASH_DATASET)
        temporario.replace(caminho_zip)
    verificar_hash(caminho_zip, HASH_DATASET)
    print(auditar_zip(caminho_zip))
    # Exemplo: extrair_wav(caminho_zip, ARQUIVO_AUDIO_NO_CSV, WAV_AUDIO)


## 5. Classificação de um WAV
O modelo compartilhado contém `modelo` e `colunas` e foi salvo com scikit-learn 1.6.1. As seis características são extraídas na mesma ordem usada no treinamento.

Para executar esta etapa, use Python 3.11 ou 3.12, forneça o modelo e um WAV e ative `EXECUTAR_AUDIO_WAV`. Se a instalação substituir uma biblioteca já importada, reinicie o ambiente e execute novamente.

Whisper tiny fornece a transcrição usada no cálculo do ritmo de fala. A transcrição Azure é analisada separadamente para manter a representação usada no treinamento do Random Forest.


In [ ]:
COLUNAS_AUDIO = ["duracao", "energia_media", "proporcao_pausas",
                 "ritmo_palavras_minuto", "tom_medio_hz", "tom_desvio_hz"]
_cache_audio = {}

def carregar_modelo_audio(caminho=MODELO_AUDIO):
    import importlib.metadata, warnings, joblib
    from sklearn.exceptions import InconsistentVersionWarning
    if importlib.metadata.version("scikit-learn") != "1.6.1":
        raise RuntimeError("Instale scikit-learn==1.6.1 e reinicie o runtime antes de carregar o modelo.")
    verificar_hash(caminho, HASH_MODELO_AUDIO)
    with warnings.catch_warnings():
        warnings.simplefilter("error", InconsistentVersionWarning)
        pacote = joblib.load(caminho)
    if not isinstance(pacote, dict) or pacote.get("colunas") != COLUNAS_AUDIO:
        raise ValueError("Esquema de características diferente do modelo auditado.")
    modelo = pacote.get("modelo")
    if set(map(str, modelo.classes_)) != CLASSES_AUDIO:
        raise ValueError("Classes do modelo diferentes do contrato.")
    return pacote

def extrair_caracteristicas_audio(caminho, transcritor):
    import librosa, numpy as np, pandas as pd
    audio, sr = librosa.load(str(caminho), sr=16000, mono=True)
    if len(audio) < 2048 or not np.isfinite(audio).all():
        raise ValueError("Áudio vazio, muito curto ou com amostras inválidas.")
    if float(np.max(np.abs(audio))) < 1e-5:
        raise ValueError("Áudio sem energia suficiente para classificar.")
    duracao = librosa.get_duration(y=audio, sr=sr)
    energia = float(np.mean(librosa.feature.rms(y=audio)[0]))
    intervalos = librosa.effects.split(audio, top_db=30)
    tempo_sonoro = sum((fim-inicio)/sr for inicio, fim in intervalos)
    texto = str(transcritor(str(caminho))).strip()
    if not texto:
        raise ValueError("Nenhuma fala transcrita; classificação acústica inconclusiva.")
    f0, _, _ = librosa.pyin(audio, fmin=70, fmax=500, sr=sr)
    f0_valido = f0[~np.isnan(f0)]
    valores = {
        "duracao": float(duracao), "energia_media": energia,
        "proporcao_pausas": max(float(duracao)-float(tempo_sonoro), 0)/duracao,
        "ritmo_palavras_minuto": len(texto.split())/(duracao/60),
        "tom_medio_hz": float(np.mean(f0_valido)) if len(f0_valido) else 0.0,
        "tom_desvio_hz": float(np.std(f0_valido)) if len(f0_valido) else 0.0,
    }
    if not all(math.isfinite(v) for v in valores.values()):
        raise ValueError("Características acústicas não finitas.")
    return pd.DataFrame([valores], columns=COLUNAS_AUDIO), texto

def transcrever_whisper(caminho):
    import whisper
    if "whisper" not in _cache_audio:
        _cache_audio["whisper"] = whisper.load_model("tiny")
    return _cache_audio["whisper"].transcribe(caminho, language="pt", fp16=False)["text"]

def classificar_audio(caminho_audio):
    if "pacote" not in _cache_audio:
        _cache_audio["pacote"] = carregar_modelo_audio()
    amostra, texto = extrair_caracteristicas_audio(caminho_audio, transcrever_whisper)
    modelo = _cache_audio["pacote"]["modelo"]
    probabilidades = modelo.predict_proba(amostra)[0]
    indice = int(probabilidades.argmax())
    _cache_audio["ultima_transcricao_whisper"] = texto
    _cache_audio["ultimas_caracteristicas"] = amostra.iloc[0].to_dict()
    return str(modelo.classes_[indice]), float(probabilidades[indice])

def executar_audio_seguro(caminho):
    try:
        classe, confianca = classificar_audio(caminho)
        return {"classe_audio": classe, "confianca_audio": confianca,
                "transcricao_whisper": _cache_audio["ultima_transcricao_whisper"],
                "caracteristicas": _cache_audio["ultimas_caracteristicas"]}
    except Exception as erro:
        # Registra a falha como indisponibilidade; não inventa classe neutra.
        return {"status": "erro", "motivo": f"Falha de áudio: {type(erro).__name__}: {erro}"}

if EXECUTAR_AUDIO_WAV:
    instalar(["scikit-learn==1.6.1", "librosa==0.11.0", "soundfile==0.13.1",
              "joblib==1.5.2", "pandas>=2.2,<3", "openai-whisper==20250625"])
    resultado_wav = executar_audio_seguro(WAV_AUDIO)
    (SAIDA / "resultado_audio_wav.json").write_text(
        json.dumps(resultado_wav, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    print(resultado_wav)
else:
    print("Inferência WAV desativada; o classificador do grupo está disponível como função.")


## 6. Análise de um vídeo
Adaptação de `video/alerta_video.py`, mantendo as classes `Tools` e `Uterus`, os limiares e o formato de resultado. Os recursos de leitura e gravação são verificados e fechados ao final.

Forneça o vídeo e `best_uterus_tools_v4.pt` para ativar `EXECUTAR_VIDEO`. Se a análise já foi feita na frente de vídeo, use o JSON gerado e pule a inferência.

A confiança de cada evento corresponde à detecção do instrumento. As métricas do detector estão no relatório da frente de vídeo.


In [ ]:
def detectar_eventos(quadros, passo_s, min_perda_s=5, tolerancia_s=1, duracao_video=None):
    passo_s = numero_finito(passo_s, "passo_s", 1e-9)
    min_perda_s = numero_finito(min_perda_s, "min_perda_s", 1e-9)
    tolerancia_s = numero_finito(tolerancia_s, "tolerancia_s", 0)
    eventos, inicio, ultimo, confs = [], None, None, []
    def fechar():
        if inicio is None:
            return
        fim = ultimo + passo_s
        if duracao_video is not None:
            fim = min(fim, duracao_video)
        if fim - inicio + 1e-8 >= min_perda_s:
            eventos.append({"descricao": "Instrumento em cena sem visualização do útero",
                            "inicio_s": round(inicio, 3), "fim_s": round(fim, 3),
                            "duracao_s": round(fim-inicio, 3),
                            "confianca": round(statistics.mean(confs), 3)})
    anterior = -1.0
    for t, instrumento, utero, conf in quadros:
        t = numero_finito(t, "instante", 0)
        if t <= anterior:
            raise ValueError("Quadros precisam estar em ordem temporal estrita.")
        anterior = t
        # Não une segmentos separados por uma lacuna de amostragem grande.
        if inicio is not None and t - ultimo > tolerancia_s + passo_s + 1e-8:
            fechar()
            inicio, ultimo, confs = None, None, []
        if instrumento and not utero:
            if inicio is None:
                inicio = t
            ultimo = t
            confs.append(numero_finito(conf, "confianca_instrumento", 0, 1))
        elif inicio is not None and t - ultimo > tolerancia_s + 1e-8:
            fechar()
            inicio, ultimo, confs = None, None, []
    fechar()
    return eventos

def nivel_do_alerta(eventos, min_perda_s=5):
    if not eventos:
        return "nenhum"
    maior = max(e["duracao_s"] for e in eventos)
    if maior >= 3*min_perda_s or len(eventos) >= 3:
        return "alto"
    return "medio" if maior >= 2*min_perda_s else "baixo"

def analisar_video(video, pesos, saida=None, conf=0.4, conf_area=0.6,
                   amostra_fps=5, min_perda_s=5, tolerancia_s=1,
                   salvar_video=True, modelo=None):
    import cv2
    from ultralytics import YOLO
    for nome, v in (("conf", conf), ("conf_area", conf_area)):
        numero_finito(v, nome, 0, 1)
    numero_finito(amostra_fps, "amostra_fps", 1e-9)
    numero_finito(min_perda_s, "min_perda_s", 1e-9)
    numero_finito(tolerancia_s, "tolerancia_s", 0)
    pasta = Path(saida or SAIDA / "video")
    pasta.mkdir(parents=True, exist_ok=True)
    if not Path(video).is_file() or (modelo is None and not Path(pesos).is_file()):
        raise FileNotFoundError("Forneça vídeo e pesos treinados nos caminhos configurados.")
    modelo = modelo if modelo is not None else YOLO(str(pesos))
    nomes = dict(enumerate(modelo.names)) if isinstance(modelo.names, list) else modelo.names
    ids = {nome: i for i, nome in nomes.items()}
    if not {"Tools", "Uterus"}.issubset(ids):
        raise ValueError("Os pesos precisam conter as classes Tools e Uterus.")
    cap = cv2.VideoCapture(str(video))
    gravador = None
    quadros, n = [], 0
    anotado = pasta / f"{Path(video).stem}_anotado.mp4"
    try:
        if not cap.isOpened():
            raise ValueError("Não foi possível abrir o vídeo.")
        fps = numero_finito(cap.get(cv2.CAP_PROP_FPS), "fps", 1e-9)
        passo = max(1, round(fps/amostra_fps))
        fps_saida = fps/passo
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if n % passo == 0:
                r = modelo.predict(frame, conf=min(conf, conf_area), verbose=False)[0]
                dets = list(zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()))
                instr = [p for k, p in dets if int(k) == ids["Tools"] and p >= conf]
                utero = any(int(k) == ids["Uterus"] and p >= conf_area for k, p in dets)
                quadros.append((n/fps, bool(instr), utero, max(instr, default=0.0)))
                if salvar_video:
                    imagem = r.plot()
                    if instr and not utero:
                        cv2.putText(imagem, "UTERO FORA DE VISTA", (10, 30),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                    if gravador is None:
                        h, w = imagem.shape[:2]
                        gravador = cv2.VideoWriter(str(anotado),
                            cv2.VideoWriter_fourcc(*"mp4v"), fps_saida, (w, h))
                        if not gravador.isOpened():
                            raise ValueError("Codec/gravador de vídeo indisponível.")
                    gravador.write(imagem)
            n += 1
        if not quadros:
            raise ValueError("Nenhum quadro foi decodificado.")
        duracao = n/fps
        eventos = detectar_eventos(quadros, passo/fps, min_perda_s, tolerancia_s, duracao)
        for i, evento in enumerate(eventos, 1):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int((evento["inicio_s"]+evento["fim_s"])/2*fps))
            ok, quadro = cap.read()
            if ok:
                destino = pasta / f"{Path(video).stem}_evento{i}.jpg"
                if cv2.imwrite(str(destino), modelo.predict(quadro, conf=conf, verbose=False)[0].plot()):
                    evento["quadro"] = str(destino)
        res = {"alerta": bool(eventos), "nivel": nivel_do_alerta(eventos, min_perda_s),
               "motivo": f"{len(eventos)} trecho(s) com instrumento sem útero visível",
               "eventos": eventos,
               "estatisticas": {"duracao_video_s": round(duracao, 3),
                   "quadros_analisados": len(quadros), "amostra_fps_efetivo": fps_saida,
                   "prop_quadros_com_utero": sum(q[2] for q in quadros)/len(quadros),
                   "prop_quadros_com_instrumento": sum(q[1] for q in quadros)/len(quadros),
                   "prop_instrumento_sem_utero": sum(q[1] and not q[2] for q in quadros)/len(quadros)},
               "parametros": {"conf": conf, "conf_area": conf_area, "min_perda_s": min_perda_s,
                              "tolerancia_s": tolerancia_s},
               "video": str(video), "video_anotado": str(anotado) if salvar_video else None,
               "aviso": AVISO}
    finally:
        cap.release()
        if gravador is not None:
            gravador.release()
    (pasta / f"resultado_{Path(video).stem}.json").write_text(
        json.dumps(res, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    return res

def executar_video_seguro(video, pesos):
    try:
        return analisar_video(video, pesos)
    except Exception as erro:
        return {"status": "erro", "motivo": f"Falha de vídeo: {type(erro).__name__}: {erro}"}

if EXECUTAR_VIDEO:
    instalar(["ultralytics>=8.3,<9", "opencv-python-headless>=4.10,<5"])
    resultado_video_arquivo = executar_video_seguro(VIDEO, PESOS_VIDEO)
    print("Resultado do vídeo:", resultado_video_arquivo.get("nivel", "indisponível"))
else:
    print("Vídeo desativado. Pode-se integrar o JSON já gerado pelo notebook do Marcelo.")


## 7. Serviços Azure
Cadastre os Secrets `SPEECH_KEY`, `SPEECH_REGION`, `AZURE_LANGUAGE_KEY` e `AZURE_LANGUAGE_ENDPOINT` no Colab e habilite o acesso do notebook. Informe o caminho de um WAV e ative `EXECUTAR_AZURE`.

O Speech transcreve um WAV PCM mono a 16 kHz. O Language analisa sentimento e frases-chave da transcrição. O vocabulário de termos para revisão é configurável; a seleção de termos ainda depende de análise de contexto.

Para testar apenas a conexão, mantenha `EXECUTAR_INTEGRACAO = False`. As saídas são `transcricao_azure.json` e `texto_azure.json`.

A execução com os serviços reais está pendente. Referências: [Speech SDK](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/get-started-speech-to-text?pivots=programming-language-python) e [Text Analytics](https://learn.microsoft.com/en-us/python/api/overview/azure/ai-textanalytics-readme?view=azure-python).


In [ ]:
def obter_segredo(nome):
    if EM_COLAB:
        from google.colab import userdata
        try:
            valor = userdata.get(nome)
        except Exception as erro:
            raise RuntimeError(
                f"Configure o Secret {nome} e habilite o acesso do notebook."
            ) from erro
    else:
        valor = os.environ.get(nome)
    if not valor:
        raise RuntimeError(f"Configure o Secret {nome}.")
    return valor

def transcrever_azure(caminho_audio):
    import azure.cognitiveservices.speech as speechsdk
    import librosa, soundfile as sf
    from threading import Event
    audio, sr = librosa.load(str(caminho_audio), sr=16000, mono=True)
    if len(audio) == 0:
        raise ValueError("Arquivo de áudio vazio.")
    caminho_pcm = ENTRADAS / "audio_azure_pcm.wav"
    sf.write(str(caminho_pcm), audio, sr, subtype="PCM_16")
    config = speechsdk.SpeechConfig(subscription=obter_segredo("SPEECH_KEY"),
                                    region=obter_segredo("SPEECH_REGION"))
    config.speech_recognition_language = "pt-BR"
    reconhecedor = speechsdk.SpeechRecognizer(
        speech_config=config, audio_config=speechsdk.audio.AudioConfig(filename=str(caminho_pcm)))
    textos, erros, segmentos = [], [], []
    terminou = Event()
    def reconhecido(evt):
        if evt.result.reason == speechsdk.ResultReason.RecognizedSpeech:
            textos.append(evt.result.text)
            segmentos.append({"inicio_s": evt.result.offset/10_000_000,
                              "duracao_s": evt.result.duration/10_000_000,
                              "texto": evt.result.text})
    def cancelado(evt):
        if evt.reason == speechsdk.CancellationReason.Error:
            erros.append(str(evt.error_code))
        terminou.set()
    reconhecedor.recognized.connect(reconhecido)
    reconhecedor.canceled.connect(cancelado)
    reconhecedor.session_stopped.connect(lambda evt: terminou.set())
    reconhecedor.start_continuous_recognition_async().get()
    try:
        if not terminou.wait(timeout=max(60, len(audio)/sr*2 + 30)):
            raise TimeoutError("Tempo limite no Azure Speech.")
    finally:
        reconhecedor.stop_continuous_recognition_async().get()
    if erros:
        raise RuntimeError("Azure Speech retornou erro: " + ", ".join(erros))
    texto = " ".join(textos).strip()
    if not texto:
        raise ValueError("Azure Speech não reconheceu fala.")
    return {"texto": texto, "segmentos": segmentos, "provedor": "azure_speech_to_text"}

def normalizar_termo(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.casefold())
                   if unicodedata.category(c) != "Mn")

def analisar_texto_azure(texto, cliente=None, termos=None):
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("Texto vazio.")
    if len(texto) > 4500:
        raise ValueError("Use trechos com até 4.500 caracteres; não há truncamento silencioso.")
    if cliente is None:
        from azure.ai.textanalytics import TextAnalyticsClient
        from azure.core.credentials import AzureKeyCredential
        cliente = TextAnalyticsClient(
            endpoint=obter_segredo("AZURE_LANGUAGE_ENDPOINT"),
            credential=AzureKeyCredential(obter_segredo("AZURE_LANGUAGE_KEY")),
            connection_timeout=10, read_timeout=30)
    sentimento = cliente.analyze_sentiment([texto], language="pt", disable_service_logs=True)[0]
    frases = cliente.extract_key_phrases([texto], language="pt", disable_service_logs=True)[0]
    if sentimento.is_error or frases.is_error:
        raise RuntimeError("Azure Text Analytics não concluiu todas as análises.")
    vocabulario = termos if termos is not None else ["dor", "falta de ar", "sangramento", "tontura"]
    criticos = sorted({termo for termo in vocabulario
                       if any(re.search(r"\b" + re.escape(normalizar_termo(termo)) + r"\b",
                                        normalizar_termo(frase))
                              for frase in frases.key_phrases)})
    scores = {"positivo": float(sentimento.confidence_scores.positive),
              "neutro": float(sentimento.confidence_scores.neutral),
              "negativo": float(sentimento.confidence_scores.negative)}
    for nome, valor in scores.items():
        numero_finito(valor, nome, 0, 1)
    sinal = bool(criticos) or (sentimento.sentiment == "negative" and scores["negativo"] >= 0.80)
    return {"modalidade": "texto_azure", "status": "ok", "alerta": sinal,
            "nivel": "medio" if sinal else "nenhum",
            "motivo": f"Sentimento textual: {sentimento.sentiment}; menções selecionadas: {len(criticos)}.",
            "sentimento": sentimento.sentiment, "scores_sentimento": scores,
            "frases_chave": list(frases.key_phrases), "termos_criticos": criticos,
            "origem": "azure_ai_language", "simulado": False,
            "ressalva": "Menções podem estar negadas ou fora de contexto; revisar a transcrição."}

if EXECUTAR_AZURE:
    instalar(["azure-cognitiveservices-speech>=1.40,<2", "azure-ai-textanalytics==5.4.0",
              "librosa==0.11.0", "soundfile==0.13.1"])
    try:
        transcricao_azure = transcrever_azure(WAV_AUDIO)
        resultado_texto_azure = analisar_texto_azure(transcricao_azure["texto"])
        (SAIDA / "transcricao_azure.json").write_text(
            json.dumps(transcricao_azure, ensure_ascii=False, indent=2), encoding="utf-8")
        (SAIDA / "texto_azure.json").write_text(
            json.dumps(resultado_texto_azure, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Speech to Text e Language concluídos; resultados salvos.")
    except Exception as erro:
        resultado_texto_azure = fonte_indisponivel(
            "texto_azure", f"Azure não concluído: {type(erro).__name__}", status="erro")
        print(resultado_texto_azure["motivo"])
else:
    print("Azure pendente: recursos e Secrets ainda não configurados.")


## 8. Séries e prescrições
Exemplos sintéticos para os tipos de anomalia previstos no enunciado:

- Séries de sinais vitais e movimento: desvio em relação à mediana e ao MAD de um histórico anterior.
- Prescrições: comparação de dose, unidade, via e intervalo de um mesmo medicamento.

Cada chamada representa um caso, com timestamps ordenados e fuso horário. O histórico de referência não deve incluir as observações avaliadas. Os limiares de seis desvios robustos e variação de dose de 50% servem à demonstração e precisam de avaliação antes de uso assistencial.


In [ ]:
def validar_instantes(registros):
    anterior = None
    for r in registros:
        instante = datetime.fromisoformat(str(r["timestamp"]).replace("Z", "+00:00"))
        if instante.tzinfo is None or (anterior is not None and instante <= anterior):
            raise ValueError("Timestamps devem ter fuso e ordem estrita crescente.")
        anterior = instante

def detectar_series(historico, observacoes, campos, modalidade, limiar=6.0, simulado=False):
    if modalidade not in {"sinais_vitais", "movimentacao"} or not campos:
        raise ValueError("Defina modalidade e campos de série.")
    numero_finito(limiar, "limiar_robusto", 0.1)
    if len(historico) < 20 or not observacoes:
        return fonte_indisponivel(modalidade, "Histórico mínimo: 20 pontos; observações: ao menos 1.",
                                 simulado=simulado)
    validar_instantes(historico + observacoes)
    referencias, eventos = {}, []
    for campo in campos:
        valores = [numero_finito(r[campo], campo) for r in historico]
        centro = statistics.median(valores)
        escala = 1.4826 * statistics.median(abs(v-centro) for v in valores)
        if escala <= 1e-12:
            return fonte_indisponivel(modalidade, f"Referência sem dispersão suficiente: {campo}.",
                                     simulado=simulado)
        referencias[campo] = {"mediana": centro, "escala_robusta": escala}
    for r in observacoes:
        for campo, ref in referencias.items():
            valor = numero_finito(r[campo], campo)
            escore = abs(valor-ref["mediana"])/ref["escala_robusta"]
            if escore >= limiar:
                eventos.append({"timestamp": r["timestamp"], "campo": campo,
                                "valor": valor, "escore_robusto": escore})
    sinal = bool(eventos)
    return {"modalidade": modalidade, "status": "ok", "alerta": sinal,
            "nivel": "medio" if sinal else "nenhum",
            "motivo": f"{len(eventos)} desvio(s) estatístico(s) em relação ao histórico.",
            "eventos": eventos, "referencias": referencias, "limiar": limiar,
            "origem": "serie_estruturada_mad", "simulado": bool(simulado)}

def detectar_prescricoes(registros, limiar_variacao=0.5, simulado=False):
    numero_finito(limiar_variacao, "limiar_variacao", 0.01)
    if len(registros) < 2:
        return fonte_indisponivel("prescricoes", "Forneça ao menos duas prescrições.",
                                 simulado=simulado)
    validar_instantes(registros)
    anteriores, eventos, comparacoes = {}, [], 0
    for r in registros:
        chave = r["medicamento_id"]
        dose = numero_finito(r["dose"], "dose", 1e-9)
        intervalo = numero_finito(r["intervalo_h"], "intervalo_h", 1e-9)
        if not chave or not r["unidade"] or not r["via"]:
            raise ValueError("Informe medicamento_id, unidade e via.")
        if chave in anteriores:
            p = anteriores[chave]
            comparacoes += 1
            motivos = []
            if r["unidade"] != p["unidade"]:
                motivos.append("unidade alterada; doses não comparadas")
            if r["via"] != p["via"]:
                motivos.append("via alterada; doses não comparadas")
            if r["unidade"] == p["unidade"] and r["via"] == p["via"]:
                variacao = abs(dose-p["dose"])/p["dose"]
                if variacao >= limiar_variacao:
                    motivos.append(f"variação relativa de dose: {variacao:.1%}")
            if intervalo != p["intervalo_h"]:
                motivos.append("intervalo entre doses alterado")
            if motivos:
                eventos.append({"timestamp": r["timestamp"], "medicamento_id": chave,
                                "motivos": motivos})
        anteriores[chave] = {**r, "dose": dose, "intervalo_h": intervalo}
    if not comparacoes:
        return fonte_indisponivel("prescricoes", "Nenhum medicamento repetido para comparar.",
                                 simulado=simulado)
    sinal = bool(eventos)
    return {"modalidade": "prescricoes", "status": "ok", "alerta": sinal,
            "nivel": "medio" if sinal else "nenhum",
            "motivo": f"{len(eventos)} alteração(ões) de prescrição para revisão.",
            "eventos": eventos, "origem": "comparacao_temporal_prescricoes",
            "simulado": bool(simulado), "limiar_variacao": limiar_variacao}


In [ ]:
# Demonstração adicional determinística; não contém dados de pacientes.
from datetime import timedelta
instante_base = datetime(2026, 1, 1, tzinfo=timezone.utc)
historico_demo = [
    {"timestamp": (instante_base+timedelta(minutes=i)).isoformat(),
     "pulso_demo": 70 + (i%5-2), "movimento_demo": 10 + (i%5-2)/2}
    for i in range(30)
]
observacoes_demo = [
    {"timestamp": (instante_base+timedelta(minutes=30)).isoformat(),
     "pulso_demo": 90, "movimento_demo": 20}
]
prescricoes_demo = [
    {"timestamp": "2026-01-01T00:00:00+00:00", "medicamento_id": "MED-DEMO",
     "dose": 10, "unidade": "unidade_demo", "via": "via_demo", "intervalo_h": 12},
    {"timestamp": "2026-01-01T00:01:00+00:00", "medicamento_id": "MED-DEMO",
     "dose": 20, "unidade": "unidade_demo", "via": "via_demo", "intervalo_h": 12},
]
fontes_estruturadas_demo = [
    detectar_series(historico_demo, observacoes_demo, ["pulso_demo"], "sinais_vitais", simulado=True),
    detectar_series(historico_demo, observacoes_demo, ["movimento_demo"], "movimentacao", simulado=True),
    detectar_prescricoes(prescricoes_demo, simulado=True),
]
resultado_estendido_demo = combinar_audio_video(
    ("neutro", 0.95), VIDEO_SEM_SINAL, id_caso="demo-estendido", simulado=True,
    pareamento="dados_sinteticos_de_um_caso_ficticio",
    fontes_adicionais=fontes_estruturadas_demo,
    fontes_esperadas=("audio", "video", "sinais_vitais", "movimentacao", "prescricoes", "texto_azure"),
)
salvar_resultados([resultado_estendido_demo], SAIDA / "demonstracao_estendida")
print("Demonstração estendida:", resultado_estendido_demo["nivel"])
print("Fonte ainda pendente:", resultado_estendido_demo["fontes_inconclusivas"])


## 9. Integração dos arquivos
`EXECUTAR_CSV_REAL` combina uma linha do CSV de áudio com o JSON de vídeo.

`EXECUTAR_INTEGRACAO` aceita os resultados dos modelos executados no notebook e usa CSV/JSON como entrada quando a inferência correspondente estiver desligada. A saída Azure é acrescentada quando `EXECUTAR_AZURE` estiver ativo.

Defina `ID_CASO_REAL`, escolha o áudio e confirme o pareamento com `PAREAMENTO_CONFIRMADO`. Quando uma fonte esperada faltar, o resultado mantém a indicação de cobertura incompleta.


In [ ]:
def integrar_arquivos_configurados(*, id_caso, pareamento_confirmado=False,
                                    csv_audio=None, arquivo_audio=None, json_video=None,
                                    resultado_audio=None, resultado_video=None,
                                    resultado_texto=None, esperar_azure=False):
    if not pareamento_confirmado:
        raise ValueError("Confirme o caso e o pareamento antes de integrar os arquivos.")

    origem_audio = "classificar_audio" if resultado_audio is not None else "nao_fornecido"
    if resultado_audio is None and csv_audio is not None and Path(csv_audio).is_file():
        resultado_audio = ler_audio_csv(csv_audio, arquivo_audio)
        origem_audio = f"CSV:{Path(csv_audio).name}:{arquivo_audio}"

    origem_video = "analisar_video" if resultado_video is not None else "nao_fornecido"
    if resultado_video is None and json_video is not None and Path(json_video).is_file():
        resultado_video = json.loads(Path(json_video).read_text(encoding="utf-8-sig"))
        origem_video = f"JSON:{Path(json_video).name}"

    adicionais = [resultado_texto] if resultado_texto is not None else []
    esperadas = ["audio", "video"]
    if esperar_azure or resultado_texto is not None:
        esperadas.append("texto_azure")

    return combinar_audio_video(
        resultado_audio, resultado_video, id_caso=id_caso,
        origem_audio=origem_audio, origem_video=origem_video,
        pareamento="arquivos_associados_por_caso_sem_sincronia_verificada",
        fontes_adicionais=adicionais, fontes_esperadas=esperadas,
    )

if EXECUTAR_CSV_REAL:
    resultado_real = integrar_csv_json(
        CSV_AUDIO, ARQUIVO_AUDIO_NO_CSV, JSON_VIDEO, id_caso=ID_CASO_REAL,
        pareamento_confirmado=PAREAMENTO_CONFIRMADO)
    salvar_resultados([resultado_real], SAIDA / "csv_json")
    print(relatorio_automatico(resultado_real))

# Os testes de áudio, vídeo e Azure podem ser executados separadamente.
if EXECUTAR_INTEGRACAO:
    resultado_arquivos = integrar_arquivos_configurados(
        id_caso=ID_CASO_REAL, pareamento_confirmado=PAREAMENTO_CONFIRMADO,
        csv_audio=CSV_AUDIO, arquivo_audio=ARQUIVO_AUDIO_NO_CSV, json_video=JSON_VIDEO,
        resultado_audio=resultado_wav if EXECUTAR_AUDIO_WAV else None,
        resultado_video=resultado_video_arquivo if EXECUTAR_VIDEO else None,
        resultado_texto=resultado_texto_azure if EXECUTAR_AZURE else None,
        esperar_azure=EXECUTAR_AZURE,
    )
    salvar_resultados([resultado_arquivos], SAIDA / "arquivos")
    print(relatorio_automatico(resultado_arquivos))
elif not EXECUTAR_CSV_REAL:
    print("Integração de arquivos desativada. Resultados da demonstração disponíveis em:", SAIDA)


## 10. Verificação
Os testes verificam contratos, limites, regras temporais, pareamento, leitura e exportação dos arquivos. O cliente Azure controlado valida o tratamento das respostas sem usar credenciais nem chamar o serviço real.


In [ ]:
from tempfile import TemporaryDirectory
from types import SimpleNamespace
testes = []

def checar(nome, condicao):
    if not condicao:
        raise AssertionError(nome)
    testes.append(nome)

def deve_falhar(nome, acao):
    try:
        acao()
    except (ValueError, TypeError, KeyError):
        testes.append(nome)
        return
    raise AssertionError("Deveria rejeitar: " + nome)

def combinar_teste(audio=("neutro", 0.95), video=None, **kwargs):
    return combinar_audio_video(audio, VIDEO_SEM_SINAL if video is None else video,
                                id_caso="teste", **kwargs)

checar("sem sinal: sem alerta", not combinar_teste()["alerta"])
checar("ansiedade com confiança: alerta", combinar_teste(("ansiedade_hesitacao", 0.81))["alerta"])
checar("tristeza com confiança: alerta", combinar_teste(("tristeza", 0.90))["alerta"])
checar("limiar inclusivo", combinar_teste(("tristeza", 0.70))["alerta"])
checar("abaixo do limiar inconclusivo", combinar_teste(("tristeza", 0.699))["status"] == "incompleto")
checar("neutro de baixa confiança pede revisão", combinar_teste(("neutro", 0.50))["requer_revisao"])
checar("vídeo alto preservado com áudio neutro", combinar_teste(video=VIDEO_COM_SINAL)["nivel"] == "alto")
checar("sinais conjuntos preservam maior nível", combinar_teste(
    ("tristeza", 0.9), VIDEO_COM_SINAL)["nivel"] == "alto")
checar("áudio ausente incompleto", combinar_teste(None)["status"] == "incompleto")
checar("vídeo ausente incompleto", combinar_audio_video(
    ("neutro", 0.95), None, id_caso="teste")["status"] == "incompleto")
checar("falha de modelo não vira neutro", combinar_teste(
    {"status": "erro", "motivo": "arquivo ilegível"})["fontes"][0]["status"] == "erro")
checar("entrada simulada não perde marca", combinar_teste(
    video={**VIDEO_SEM_SINAL, "simulado": True})["simulado"])
checar("ID determinístico", combinar_teste()["id_resultado"] == combinar_teste()["id_resultado"])
checar("fonte prevista ausente", "texto_azure" in combinar_teste(
    fontes_esperadas=("audio", "video", "texto_azure"))["fontes_inconclusivas"])
for p in (-0.1, 1.1, float("nan"), float("inf"), True):
    deve_falhar(f"confiança inválida {p}", lambda p=p: normalizar_audio(("neutro", p)))
deve_falhar("classe desconhecida", lambda: normalizar_audio(("outra", 0.9)))
deve_falhar("booleano como texto no vídeo", lambda: normalizar_video({"alerta": "false"}))
deve_falhar("vídeo inconsistente", lambda: normalizar_video({"alerta": False, "nivel": "alto"}))
deve_falhar("evento temporal invertido", lambda: normalizar_video({
    "alerta": True, "nivel": "alto", "eventos": [{"inicio_s": 3, "fim_s": 2}]}))
deve_falhar("ID inseguro", lambda: combinar_audio_video(None, None, id_caso="../caso"))
checar("evento curto rejeitado", not detectar_eventos(
    [(i, True, False, 0.9) for i in range(4)], 1))
checar("evento no fim preservado", len(detectar_eventos(
    [(i, True, False, 0.9) for i in range(5)], 1)) == 1)
checar("lacuna grande não une segmentos", not detectar_eventos(
    [(i, True, False, 0.9) for i in [0, 1, 2, 10, 11, 12]], 1))
checar("final limitado à duração", not detectar_eventos(
    [(i, True, False, 0.9) for i in range(5)], 1, duracao_video=4.5))
deve_falhar("quadros fora de ordem", lambda: detectar_eventos(
    [(1, True, False, 0.9), (0, True, False, 0.9)], 1))
checar("série sintética tem desvio", fontes_estruturadas_demo[0]["alerta"])
checar("histórico insuficiente inconclusivo", detectar_series(
    historico_demo[:2], observacoes_demo, ["pulso_demo"], "sinais_vitais")["status"] == "indisponivel")
checar("referência constante inconclusiva", detectar_series(
    [{**r, "pulso_demo": 70} for r in historico_demo], observacoes_demo,
    ["pulso_demo"], "sinais_vitais")["status"] == "indisponivel")
checar("prescrição alterada", fontes_estruturadas_demo[2]["alerta"])
checar("prescrição sem mudança", not detectar_prescricoes(
    [prescricoes_demo[0], {**prescricoes_demo[1], "dose": 10}])["alerta"])
checar("mudança de unidade não compara doses", any("unidade" in m for m in detectar_prescricoes(
    [prescricoes_demo[0], {**prescricoes_demo[1], "unidade": "outra"}])["eventos"][0]["motivos"]))
deve_falhar("série com futuro na referência", lambda: detectar_series(
    historico_demo, [historico_demo[0]], ["pulso_demo"], "sinais_vitais"))

class ClienteAzureControlado:
    def __init__(self, erro=False):
        self.erro = erro
    def analyze_sentiment(self, docs, **kwargs):
        return [SimpleNamespace(is_error=self.erro, sentiment="negative",
            confidence_scores=SimpleNamespace(positive=0.01, neutral=0.04, negative=0.95))]
    def extract_key_phrases(self, docs, **kwargs):
        return [SimpleNamespace(is_error=self.erro, key_phrases=["falta de ar"])]

texto_controlado = analisar_texto_azure("Relato demonstrativo.", cliente=ClienteAzureControlado())
checar("adaptador Azure usa sentimento e frases-chave", texto_controlado["alerta"] and
       texto_controlado["termos_criticos"] == ["falta de ar"])
try:
    analisar_texto_azure("Texto.", cliente=ClienteAzureControlado(erro=True))
except RuntimeError:
    testes.append("erro de documento Azure não vira sucesso")
else:
    raise AssertionError("Erro Azure ignorado.")

with TemporaryDirectory() as d:
    p = Path(d)
    csv_teste = p / "audio.csv"
    csv_teste.write_text("arquivo,classe_audio,confianca_audio\nfala.wav,neutro,0.95\n", encoding="utf-8")
    checar("CSV válido", ler_audio_csv(csv_teste, "fala.wav")["classe_audio"] == "neutro")
    deve_falhar("CSV arquivo ausente", lambda: ler_audio_csv(csv_teste, "outro.wav"))
    (p / "video.json").write_text(json.dumps(VIDEO_SEM_SINAL), encoding="utf-8")
    deve_falhar("pareamento não confirmado", lambda: integrar_csv_json(
        csv_teste, "fala.wav", p / "video.json", id_caso="teste"))
    integrado = integrar_arquivos_configurados(
        id_caso="teste-arquivos", pareamento_confirmado=True,
        csv_audio=csv_teste, arquivo_audio="fala.wav", json_video=p / "video.json")
    checar("origens do CSV e JSON preservadas",
           integrado["fontes"][0]["origem"] == "CSV:audio.csv:fala.wav"
           and integrado["fontes"][1]["origem"] == "JSON:video.json")
    integrado_azure = integrar_arquivos_configurados(
        id_caso="teste-azure", pareamento_confirmado=True,
        csv_audio=csv_teste, arquivo_audio="fala.wav", json_video=p / "video.json",
        resultado_texto={**texto_controlado, "simulado": True}, esperar_azure=True)
    checar("Azure complementa os resultados dos arquivos",
           integrado_azure["alerta"] and integrado_azure["status"] == "completo"
           and integrado_azure["simulado"])
    azure_ausente = integrar_arquivos_configurados(
        id_caso="teste-azure-ausente", pareamento_confirmado=True,
        csv_audio=csv_teste, arquivo_audio="fala.wav", json_video=p / "video.json",
        esperar_azure=True)
    checar("Azure esperado sem resultado mantém cobertura incompleta",
           azure_ausente["status"] == "incompleto"
           and "texto_azure" in azure_ausente["fontes_inconclusivas"])
    deve_falhar("integração de arquivos exige pareamento",
                lambda: integrar_arquivos_configurados(id_caso="teste-arquivos"))
    salvar_resultados(resultados_demo, p)
    salvar_resultados(resultados_demo, p)
    checar("exportação não duplica lote", len((p / "resultados.jsonl").read_text(
        encoding="utf-8").splitlines()) == len(resultados_demo))
    checar("fila inclui fonte ausente", len((p / "fila_revisao.jsonl").read_text(
        encoding="utf-8").splitlines()) == 3)
    checar("JSON preserva simulação", json.loads((p / "demo-03.json").read_text(
        encoding="utf-8"))["simulado"])
    checar("relatório identifica prioridade", "**Prioridade:** alto" in (p / "demo-03.md").read_text(
        encoding="utf-8"))

evidencia = {"testes_aprovados": len(testes), "nomes": testes,
            "chamada_azure_real_nestes_testes": False, "inferencia_yolo_real_executada_nestes_testes": False,
            "escopo": "contratos, regras, arquivos e cliente Azure controlado",
            "commit_base": COMMIT_BASE}
(SAIDA / "validacao_contratos.json").write_text(
    json.dumps(evidencia, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{len(testes)} verificações aprovadas. Serviços Azure reais não foram chamados.")


## 11. Exportação
Cada execução gera JSON, relatório por caso, resumo CSV e fila de revisão. Ative `BAIXAR_SAIDAS` para baixar os resultados no Colab.

Para a apresentação, registre quais fontes foram usadas, quais dados são simulados e os resultados das chamadas Azure após a configuração dos recursos.


In [ ]:
BAIXAR_SAIDAS = False
if BAIXAR_SAIDAS:
    import shutil
    arquivo_saida = shutil.make_archive(str(RAIZ / "resultados_integracao"), "zip", SAIDA)
    if EM_COLAB:
        from google.colab import files
        files.download(arquivo_saida)
    else:
        print(arquivo_saida)
